# 02 - ETL: Capa Bronce a Capa Plata
**Proyecto:** food-price-forecasting-peru | Big Data Analytics UNMSM 2025-II

**Descripción:** Este notebook realiza la extracción de datos crudos desde Google Cloud Storage (Zona Bronce), ejecuta la limpieza, el mapeo regional y el Feature Engineering inicial, guardando los resultados integrados en formato Parquet (Zona Plata).

**Flujo:**
1. Setup Spark con GCS Connector.
2. Ingesta de Precios (SISAP) y Clima (SENAMHI) desde GCS.
3. Limpieza y Normalización.
4. Join por Región-Semana.
5. Generación de Lags y Variables Predictoras.
6. Persistencia en Zona Plata.

In [27]:
import os
from google.colab import auth
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 1. Configuracion de variables del proyecto
BUCKET_NAME  = "food-price-peru-2025-01"
BRONCE_SISAP = f"gs://{BUCKET_NAME}/bronce/sisap/"
BRONCE_CLIMA = f"gs://{BUCKET_NAME}/bronce/senamhi/"
PLATA_PATH   = f"gs://{BUCKET_NAME}/plata/precios_clima_integrado"

print("-" * 50)
print(f"Bucket  : {BUCKET_NAME}")
print(f"SISAP   : {BRONCE_SISAP}")
print(f"SENAMHI : {BRONCE_CLIMA}")
print("-" * 50)

# 2. Autenticacion y Setup de Spark con conector GCS
auth.authenticate_user()

jar_name = "gcs-connector.jar"
jar_path = os.path.abspath(jar_name)
if not os.path.exists(jar_path):
    !wget -q https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.14/gcs-connector-hadoop3-2.2.14-shaded.jar -O {jar_name}

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {jar_path} pyspark-shell"

spark = (SparkSession.builder
    .appName("02_ETL_Bronce_a_Plata")
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.gs.project.id", BUCKET_NAME)
    .config("spark.driver.extraClassPath", jar_path)
    .config("spark.executor.extraClassPath", jar_path)
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark iniciado correctamente.")

--------------------------------------------------
Bucket  : food-price-peru-2025-01
SISAP   : gs://food-price-peru-2025-01/bronce/sisap/
SENAMHI : gs://food-price-peru-2025-01/bronce/senamhi/
--------------------------------------------------
Spark iniciado correctamente.


In [28]:
MAPEO_PRODUCTOS = {
    'Ajo': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'AREQUIPA_MAJES'),
    'Ajo Criollo': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'AREQUIPA_MAJES'),
    'Ajo Criollo O Napuri': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'AREQUIPA_MAJES'),
    'Cebolla': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'AREQUIPA_MAJES'),
    'Cebolla Cabeza Roja': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'AREQUIPA_MAJES'),
    'Cebolla Roja': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'AREQUIPA_MAJES'),
    'Papa': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'JUNIN_JAUJA'),
    'Papa Blanca': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'JUNIN_JAUJA'),
    'Papa Canchan': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'JUNIN_JAUJA'),
    'Papa Amarilla': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'HUANUCO'),
    'Papa Yungay': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'HUANUCO'),
    'Choclo': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'JUNIN_JAUJA'),
    'Tomate': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'LALIBERTAD_TRUJILLO'),
    'Vainita': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'LALIBERTAD_TRUJILLO'),
    'Vainita Americana': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'LALIBERTAD_TRUJILLO'),
    'Zanahoria': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'LALIBERTAD_TRUJILLO'),
    'Pimiento': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'LALIBERTAD_TRUJILLO'),
    'Pimiento Morron': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'LALIBERTAD_TRUJILLO'),
    'Palta': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'ICA_PALPA'),
    'Palta Criolla': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'ICA_PALPA'),
    'Palta Fuerte': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'ICA_PALPA'),
    'Limon': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'PIURA_SAPILLICA'),
    'Limon Sutil': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'PIURA_SAPILLICA'),
    'Limon Sutil Bolsa': ('Gran mercado mayorista de lima', 'Santa Anita', '150137', 'PIURA_SAPILLICA'),
    'Camote': ('Mcdo prod. santa anita', 'Santa Anita', '150137', 'ICA_PALPA'),
    'Camote Morado': ('Mcdo prod. santa anita', 'Santa Anita', '150137', 'ICA_PALPA'),
    'Yuca': ('Mcdo prod. santa anita', 'Santa Anita', '150137', 'ICA_PALPA'),
    'Yuca Amarilla': ('Mcdo prod. santa anita', 'Santa Anita', '150137', 'ICA_PALPA'),
    'Arroz': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'PIURA_SAPILLICA'),
    'Arroz Corriente': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'PIURA_SAPILLICA'),
    'Arroz Extra': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'PIURA_SAPILLICA'),
    'Arroz Superior': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'PIURA_SAPILLICA'),
    'Frijol': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Frijol Canario': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Garbanzo': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Lenteja': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Fideos': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Fideos Molitalia': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Fideos Molitalia Spaghetti': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Leche': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Leche Fresca': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Leche Fresca Gloria': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Leche Fresca Gloria En Bolsa': ('Mcdo. may. cereales, legum. y oleag.', 'Santa Anita', '150137', 'SERIE_TIEMPO'),
    'Papaya': ('Mercado mayorista nro 2-frutas', 'La Victoria', '150115', 'SANMARTIN_TARAPOTO'),
    'Maracuya': ('Mercado mayorista nro 2-frutas', 'La Victoria', '150115', 'SANMARTIN_TARAPOTO'),
    'Piña': ('Mercado mayorista nro 2-frutas', 'La Victoria', '150115', 'LALIBERTAD_TRUJILLO'),
    'Piña Criolla': ('Mercado mayorista nro 2-frutas', 'La Victoria', '150115', 'LALIBERTAD_TRUJILLO'),
    'Platano': ('Mcdo. coop. platanos', 'Cercado de Lima', '150101', 'SANMARTIN_TARAPOTO'),
    'Platano Bellaco': ('Mcdo. coop. platanos', 'Cercado de Lima', '150101', 'SANMARTIN_TARAPOTO'),
    'Platano Isla': ('Mcdo. coop. platanos', 'Cercado de Lima', '150101', 'SANMARTIN_TARAPOTO'),
    'Platano Seda': ('Mcdo. coop. platanos', 'Cercado de Lima', '150101', 'SANMARTIN_TARAPOTO'),
    'Platano Seda Congo': ('Mcdo. coop. platanos', 'Cercado de Lima', '150101', 'SANMARTIN_TARAPOTO'),
    'Pollo': ('Mercado mayorista de aves vivas', 'San Luis', '150132', 'SERIE_TIEMPO'),
    'Pollo Vivo': ('Mercado mayorista de aves vivas', 'San Luis', '150132', 'SERIE_TIEMPO'),
    'Huevos': ('Mercado mayorista de aves vivas', 'San Luis', '150132', 'SERIE_TIEMPO'),
    'Huevos Rosados': ('Mercado mayorista de aves vivas', 'San Luis', '150132', 'SERIE_TIEMPO'),
}

def match_producto(nombre):
    nombre = str(nombre).strip().title()
    if nombre in MAPEO_PRODUCTOS:
        return MAPEO_PRODUCTOS[nombre]
    for key in MAPEO_PRODUCTOS:
        if key.lower() in nombre.lower() or nombre.lower() in key.lower():
            return MAPEO_PRODUCTOS[key]
    return None

print(f'Mapeo cargado: {len(MAPEO_PRODUCTOS)} productos')


Mapeo cargado: 56 productos


In [29]:
# ETL SISAP - Extraccion desde GCS y limpieza con Pandas
import glob
import os
import pandas as pd

# 1. Carpeta temporal en la maquina local de Colab
LOCAL_SISAP = '/content/tmp_sisap'
os.makedirs(LOCAL_SISAP, exist_ok=True)

# 2. Copiamos de GCS a local para poder procesar los .xls
# BRONCE_SISAP ya apunta a gs://food-price-peru-2025-01/bronce/sisap/
print(f'Descargando de GCS: {BRONCE_SISAP}')
!gsutil -m cp -r {BRONCE_SISAP}* {LOCAL_SISAP}

def leer_sisap_robusto(file_path):
    try:
        tables = pd.read_html(file_path)
        for i, df in enumerate(tables):
            for row_idx in range(min(10, len(df))):
                row_vals = df.iloc[row_idx].astype(str).str.lower().tolist()
                if any('fecha' in v or 'precio' in v for v in row_vals):
                    df_clean = df.iloc[row_idx+1:].copy()
                    df_clean.columns = df.iloc[row_idx]
                    df_clean = df_clean.reset_index(drop=True)
                    df_clean.columns = [str(c).strip().lower() for c in df_clean.columns]

                    col_map = {}
                    for col in df_clean.columns:
                        cs = str(col).lower()
                        if any(x in cs for x in ['fecha', 'date']): col_map[col] = 'fecha'
                        elif any(x in cs for x in ['precio', 'promedio', 'soles', 's/.']): col_map[col] = 'precio'
                    df_clean = df_clean.rename(columns=col_map)

                    if 'fecha' in df_clean.columns and 'precio' in df_clean.columns:
                        producto = None
                        for r in range(min(5, len(df))):
                            for val in df.iloc[r].astype(str):
                                vs = str(val).strip()
                                if vs and len(vs) > 2 and vs not in ['nan', 'Precio Promedio']:
                                    if not any(x in vs.lower() for x in ['fecha', 'precio', 'promedio', 'lima']):
                                        producto = vs
                                        break
                            if producto: break

                        df_clean['producto'] = producto if producto else 'Desconocido'
                        df_clean['archivo_origen'] = os.path.basename(file_path)
                        df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()]

                        return df_clean[['fecha', 'precio', 'producto', 'archivo_origen']]
        return None
    except Exception as e:
        return None

sisap_files = glob.glob(os.path.join(LOCAL_SISAP, '**/*.xls'), recursive=True)
dfs_sisap = []

for f in sisap_files:
    df_temp = leer_sisap_robusto(f)
    if df_temp is not None:
        dfs_sisap.append(df_temp)

if dfs_sisap:
    df_sisap_raw = pd.concat(dfs_sisap, ignore_index=True)
    print("-" * 50)
    print("VERIFICACION DE CARGA SISAP:")
    print(f"Total registros cargados : {len(df_sisap_raw):,}")
    print(f"Archivos procesados      : {df_sisap_raw['archivo_origen'].nunique()}")
    print(f"Productos detectados     : {df_sisap_raw['producto'].nunique()}")
    print("Listado de productos (muestra):", df_sisap_raw['producto'].unique()[:5].tolist())
    print("-" * 50)
else:
    print('Error: No se pudo extraer data valida de los archivos .xls en GCS.')

Descargando de GCS: gs://food-price-peru-2025-01/bronce/sisap/
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte (1).xls...
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte (2).xls...
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte (3).xls...
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte (7).xls...
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte (5).xls...
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte.xls...
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte (6).xls...
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte (8).xls...
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte (4).xls...
Copying gs://food-price-peru-2025-01/bronce/sisap/reporte (9)

In [38]:
print(f"Buscando archivos CSV en: {BRONCE_CLIMA}")
!gsutil ls {BRONCE_CLIMA}*.csv

🔍 Buscando archivos CSV en: gs://food-price-peru-2025-01/bronce/senamhi/
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://food-price-peru-2025-01/bronce/senamhi/senamhi_arequipa_majes_2015_2024.csv
gs://food-price-peru-2025-01/bronce/senamhi/senamhi_huanuco_2015_2024.csv
gs://food-price-peru-2025-01/bronce/senamhi/senamhi_ica_palpa_2015_2024.csv
gs://food-price-peru-2025-01/bronce/senamhi/senamhi_junin_jauja_2015_2024.csv
gs://food-price-peru-2025-01/bronce/senamhi/senamhi_lalibertad_trujillo_2015_2024.csv
gs://food-price-peru-2025-01/bronce/senamhi/senamhi_piura_sapillica_2015_2024.csv
gs://food-price-peru-2025-01/bronce/senamhi/senamhi_sanmartin_tarapoto_2015_2024.csv


In [41]:
# Limpiar fechas y precios
df_sisap_raw['fecha'] = pd.to_datetime(df_sisap_raw['fecha'], errors='coerce', dayfirst=True)
df_sisap_raw['precio'] = (df_sisap_raw['precio']
                          .astype(str)
                          .str.replace(',', '.', regex=False)
                          .str.replace(' ', '')
                          .str.replace('S/.', '', regex=False)
                          .str.replace('s/.', '', regex=False))
df_sisap_raw['precio'] = pd.to_numeric(df_sisap_raw['precio'], errors='coerce')
df_sisap_raw['producto'] = df_sisap_raw['producto'].astype(str).str.strip().str.title()

# Filtrar válidos
df_sisap = df_sisap_raw[
    df_sisap_raw['fecha'].notna() &
    df_sisap_raw['precio'].notna() &
    (df_sisap_raw['precio'] > 0)
].copy()

# Aplicar mapeo
mapeo_resultados = []
for prod in df_sisap['producto'].unique():
    res = match_producto(prod)
    if res:
        mapeo_resultados.append({
            'producto': prod,
            'mercado_sisap': res[0],
            'distrito_lima': res[1],
            'ubigeo_inei': res[2],
            'region_clima_origen': res[3]
        })
    else:
        print(f'Sin mapeo: {prod}')

df_mapeo = pd.DataFrame(mapeo_resultados)
df_sisap_mapped = df_sisap.merge(df_mapeo, on='producto', how='inner')

# Features temporales
df_sisap_mapped['anio'] = df_sisap_mapped['fecha'].dt.year
df_sisap_mapped['mes'] = df_sisap_mapped['fecha'].dt.month
df_sisap_mapped['semana'] = df_sisap_mapped['fecha'].dt.isocalendar().week.astype(int)
df_sisap_mapped['dia_semana'] = df_sisap_mapped['fecha'].dt.dayofweek + 1
df_sisap_mapped['trimestre'] = df_sisap_mapped['fecha'].dt.quarter

print(f'SISAP mapeado: {len(df_sisap_mapped):,} registros')
print(f'Regiones climáticas: {df_sisap_mapped["region_clima_origen"].unique().tolist()}')

# Guardar CSV intermedio en GCS
csv_sisap = os.path.join(PLATA_PATH, 'precios_sisap_limpios.csv')
df_sisap_mapped.to_csv(csv_sisap, index=False)
print(f'CSV guardado en GCS: {csv_sisap}')

# Guardar Parquet particionado con Spark en GCS
df_sisap_spark = spark.createDataFrame(df_sisap_mapped)
parquet_sisap = os.path.join(PLATA_PATH, 'precios_spark')
df_sisap_spark.write.mode('overwrite').partitionBy('anio', 'mercado_sisap').parquet(parquet_sisap)
print(f'Parquet guardado en GCS: {parquet_sisap}')

SISAP mapeado: 34,969 registros
Regiones climáticas: ['ICA_PALPA', 'HUANUCO', 'SANMARTIN_TARAPOTO', 'SERIE_TIEMPO', 'AREQUIPA_MAJES', 'LALIBERTAD_TRUJILLO']
CSV guardado en GCS: gs://food-price-peru-2025-01/plata/precios_clima_integrado/precios_sisap_limpios.csv
Parquet guardado en GCS: gs://food-price-peru-2025-01/plata/precios_clima_integrado/precios_spark


In [52]:
# ETL SENAMHI - Procesar regiones productoras (excluyendo Lima)
import os
import pandas as pd
import numpy as np
import glob

# 1. Rutas GCS
BRONCE_CLIMA_GCS = f'gs://{BUCKET_NAME}/bronce/senamhi/'
PLATA_PATH_GCS = f'gs://{BUCKET_NAME}/plata/precios_clima_integrado/'

# 2. Carpeta temporal local
LOCAL_CLIMA = '/content/tmp_senamhi'
os.makedirs(LOCAL_CLIMA, exist_ok=True)

# 3. Descarga
print(f'Descargando de GCS: {BRONCE_CLIMA_GCS}')
!gsutil -m cp -r {BRONCE_CLIMA_GCS}*.csv {LOCAL_CLIMA}

senamhi_files = glob.glob(os.path.join(LOCAL_CLIMA, '*.csv'))
senamhi_files = [f for f in senamhi_files if 'lima' not in f.lower()]

dfs_clima = []
for file in senamhi_files:
    basename = os.path.basename(file)
    region = basename.replace('senamhi_', '').replace('_2015_2024.csv', '').upper()

    try:
        df_c = pd.read_csv(file, encoding='utf-8', sep=None, engine='python')
    except:
        df_c = pd.read_csv(file, encoding='latin-1', sep=None, engine='python')

    df_c.columns = [str(c).strip() for c in df_c.columns]

    # --- MAPEADO EXHAUSTIVO SOLICITADO ---
    col_rename = {}
    for col in df_c.columns:
        cc = col.lower()
        if 'aã±o' in cc or 'año' in cc or 'anio' in cc:
            col_rename[col] = 'anio'
        elif 'mes' in cc:
            col_rename[col] = 'mes'
        elif 'dã­a' in cc or 'día' in cc or 'dia' in cc:
            col_rename[col] = 'dia'
        elif 'mã¡x' in cc or 'máx' in cc or 'max' in cc:
            col_rename[col] = 'temp_max'
        elif 'mã­n' in cc or 'mín' in cc or 'min' in cc:
            col_rename[col] = 'temp_min'
        elif 'humedad' in cc:
            col_rename[col] = 'humedad'
        elif 'precipitaciã³n' in cc or 'precipitación' in cc or 'precipitacion' in cc:
            col_rename[col] = 'precipitacion'

    df_c = df_c.rename(columns=col_rename)

    if not all(col in df_c.columns for col in ['anio', 'mes', 'dia']):
        continue

    # Fechas
    df_fechas = df_c[['anio', 'mes', 'dia']].rename(columns={'anio': 'year', 'mes': 'month', 'dia': 'day'})
    df_c['fecha'] = pd.to_datetime(df_fechas, errors='coerce')

    # Limpieza
    for col in ['temp_max', 'temp_min', 'humedad', 'precipitacion']:
        if col in df_c.columns:
            df_c[col] = df_c[col].replace(['S/D', 's/d', 'SD', 'T', 't'], np.nan)
            df_c[col] = pd.to_numeric(df_c[col], errors='coerce')

    df_c['region_clima'] = region
    df_c['temp_avg'] = df_c[['temp_max', 'temp_min']].mean(axis=1)
    df_c['flag_helada'] = (df_c['temp_min'] < 2).astype(int)
    df_c['flag_lluvia_intensa'] = (df_c['precipitacion'] > 10).astype(int)
    df_c['semana'] = df_c['fecha'].dt.isocalendar().week.astype(int)
    df_c['anio_isocalendar'] = df_c['fecha'].dt.isocalendar().year.astype(int)

    dfs_clima.append(df_c)
    print(f'OK: {region}')

if dfs_clima:
    df_clima_all = pd.concat(dfs_clima, ignore_index=True)
    local_out = '/content/clima_completo.csv'
    df_clima_all.to_csv(local_out, index=False)
    !gsutil cp {local_out} {PLATA_PATH_GCS}clima_completo.csv
    print(f'Total consolidado: {len(df_clima_all):,}')

Descargando de GCS: gs://food-price-peru-2025-01/bronce/senamhi/
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://food-price-peru-2025-01/bronce/senamhi/senamhi_arequipa_majes_2015_2024.csv...
Copying gs://food-price-peru-2025-01/bronce/senamhi/senamhi_huanuco_2015_2024.csv...
Copying gs://food-price-peru-2025-01/bronce/senamhi/senamhi_piura_sapillica_2015_2024.csv...
Copying gs://food-price-peru-2025-01/bronce/senamhi/senamhi_sanmartin_tarapoto_2015_2024.csv...
Copying gs://food-price-peru-2025-01/bronce/senamhi/senamhi_junin_jauja_2015_2024.csv...
Copying gs://food-price-peru-2025-01/bronce/senamhi/senamhi_ica_palpa_2015_2024.csv...
Copying gs://food-price-peru-2025-01/bronce/senamhi/senamhi_lalibertad_trujillo_2015_2024.csv...
- [7/7 files][548.5 KiB/548.5 KiB] 10

/tmp/ipykernel_5642/1087291441.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c[col] = df_c[col].replace(['S/D', 's/d', 'SD', 'T', 't'], np.nan)


OK: PIURA_SAPILLICA
OK: AREQUIPA_MAJES
OK: HUANUCO
OK: ICA_PALPA
OK: LALIBERTAD_TRUJILLO
OK: JUNIN_JAUJA
OK: SANMARTIN_TARAPOTO
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file:///content/clima_completo.csv [Content-Type=text/csv]...
/ [1 files][  1.3 MiB/  1.3 MiB]                                                
Operation completed over 1 objects/1.3 MiB.                                      
Total consolidado: 18,841


In [58]:
# Agregar precios por producto-semana-región
precios_sem = df_sisap_mapped.groupby(
    ['producto', 'region_clima_origen', 'anio', 'semana']
).agg(
    precio_promedio=('precio', 'mean'),
    precio_min=('precio', 'min'),
    precio_max=('precio', 'max'),
    precio_std=('precio', 'std'),
    n_obs=('precio', 'count'),
    mercado_sisap=('mercado_sisap', 'first'),
    distrito_lima=('distrito_lima', 'first'),
    ubigeo_inei=('ubigeo_inei', 'first')
).reset_index()

# Agregar clima por región-semana
clima_sem = df_clima_all.groupby(
    ['region_clima', 'anio_isocalendar', 'semana']
).agg(
    temp_avg=('temp_avg', 'mean'),
    temp_max=('temp_max', 'mean'),
    temp_min=('temp_min', 'mean'),
    precip_total=('precipitacion', 'sum'),
    humedad_avg=('humedad', 'mean'),
    dias_helada=('flag_helada', 'sum'),
    dias_lluvia_intensa=('flag_lluvia_intensa', 'sum'),
    n_obs_clima=('fecha', 'count')
).reset_index()

# Renombrar para merge
clima_sem = clima_sem.rename(columns={'region_clima': 'region_clima_origen', 'anio_isocalendar': 'anio'})

# Join
df_joined = precios_sem.merge(
    clima_sem,
    on=['region_clima_origen', 'anio', 'semana'],
    how='left'
)

# Indicadores de validación solicitados
print(f'Join completado: {len(df_joined):,} registros')
print(f'Precios semanales: {len(precios_sem):,} | Clima semanal: {len(clima_sem):,}')
print(f'Registros con clima completo: {df_joined["temp_avg"].notna().sum():,}')

try:
    # Convertimos a Spark para aprovechar el particionamiento y formato columnar
    df_joined_spark = spark.createDataFrame(df_joined)

    parquet_path = os.path.join(PLATA_PATH_GCS, 'precios_clima_parquet')

    # Guardamos particionando por anio y producto para mejorar consultas futuras
    df_joined_spark.write.mode('overwrite')\
        .partitionBy('anio', 'producto')\
        .parquet(parquet_path)

    print(f'Dataset integrado guardado exitosamente en PARQUET en GCS')
    print(f'Ruta: {parquet_path}')
except Exception as e:
    print(f'Error al guardar en Parquet: {e}')

Join completado: 5,225 registros
Precios semanales: 5,225 | Clima semanal: 2,707
Registros con clima completo: 2,345
Dataset integrado guardado exitosamente en PARQUET en GCS
Ruta: gs://food-price-peru-2025-01/plata/precios_clima_integrado/precios_clima_parquet


In [59]:
# Feature Engineering e Imputación Climática Avanzada en Spark

# 1. Convertir a Spark el Join original (mantiene todo el histórico)
df_spark_base = spark.createDataFrame(df_joined)

# 2. Definir ventanas para la imputación avanzada (LOCF y NOCB) por región y cronología
window_pasado = Window.partitionBy('region_clima_origen').orderBy('anio', 'semana').rowsBetween(Window.unboundedPreceding, 0)
window_futuro = Window.partitionBy('region_clima_origen').orderBy('anio', 'semana').rowsBetween(0, Window.unboundedFollowing)

# 3. Aplicar Imputación Avanzada sobre las variables de SENAMHI
# Si hay un NaN, busca el último dato válido hacia atrás; si no hay, busca hacia adelante.
df_clima_imputado = df_spark_base
for col_clima in ['temp_avg', 'temp_max', 'temp_min', 'precip_total', 'humedad_avg']:
    df_clima_imputado = df_clima_imputado.withColumn(
        col_clima,
        F.coalesce(
            F.last(col_clima, ignorenulls=True).over(window_pasado), # MIRA AL PASADO (LOCF)
            F.first(col_clima, ignorenulls=True).over(window_futuro)  # MIRA AL FUTURO (NOCB)
        )
    )

# 4. Definir la ventana analítica por producto para los Lags exigidos en la guía (pág. 9, 13)
window_producto = Window.partitionBy('producto').orderBy('anio', 'semana')

# 5. Generar las variables predictoras limpias
df_features_calculados = (df_clima_imputado
    # Lags de precio requeridos por la arquitectura del proyecto (págs. 9, 13)
    .withColumn('precio_lag_1', F.lag('precio_promedio', 1).over(window_producto))
    .withColumn('precio_lag_2', F.lag('precio_promedio', 2).over(window_producto))
    .withColumn('precio_lag_4', F.lag('precio_promedio', 4).over(window_producto))

    # Medias móviles y volatilidad
    .withColumn('precio_ma_4', F.avg('precio_promedio').over(window_producto.rowsBetween(-3, 0)))
    .withColumn('precio_ma_8', F.avg('precio_promedio').over(window_producto.rowsBetween(-7, 0)))
    .withColumn('precio_std_4', F.stddev('precio_promedio').over(window_producto.rowsBetween(-3, 0)))
    .withColumn('variacion_1', (F.col('precio_promedio') - F.col('precio_lag_1')) / F.col('precio_lag_1'))

    # Variables de calendario
    .withColumn('mes', F.when(F.col('semana') <= 4, 1)
                       .when(F.col('semana') <= 8, 2)
                       .when(F.col('semana') <= 13, 3)
                       .when(F.col('semana') <= 17, 4)
                       .when(F.col('semana') <= 21, 5)
                       .when(F.col('semana') <= 26, 6)
                       .when(F.col('semana') <= 30, 7)
                       .when(F.col('semana') <= 35, 8)
                       .when(F.col('semana') <= 39, 9)
                       .when(F.col('semana') <= 43, 10)
                       .when(F.col('semana') <= 48, 11)
                       .otherwise(12))
    .withColumn('trimestre', F.ceil(F.col('mes') / 3))

    # Variables objetivo a 2 semanas (Forecasting lead)
    .withColumn('target_precio_2s', F.lead('precio_promedio', 2).over(window_producto))
    .withColumn('target_alerta_10', F.when((F.col('target_precio_2s') - F.col('precio_promedio')) / F.col('precio_promedio') > 0.10, 1).otherwise(0))
)

# 6. FILTRO DE FECHAS ESTRICTO
# Recortamos para el rango oficial asegurando que no queden registros flotantes sin target o vacíos
df_features = df_features_calculados.filter(
    (F.col('anio') >= 2017) &
    (F.col('precio_lag_1').isNotNull()) &
    (F.col('precio_lag_2').isNotNull())
)

print(f'📊 Matriz con Imputación Avanzada Completada: {df_features.count():,} registros')
df_features.select('producto', 'anio', 'semana', 'precio_promedio', 'precio_lag_1', 'temp_avg', 'precip_total', 'target_precio_2s').show(80, truncate=False)

📊 Matriz con Imputación Avanzada Completada: 4,697 registros
+--------------------+----+------+------------------+------------------+------------------+------------------+------------------+
|producto            |anio|semana|precio_promedio   |precio_lag_1      |temp_avg          |precip_total      |target_precio_2s  |
+--------------------+----+------+------------------+------------------+------------------+------------------+------------------+
|Ajo Criollo O Napuri|2017|1     |5.198571428571428 |2.815             |20.849999999999998|0.0               |4.9628571428571435|
|Ajo Criollo O Napuri|2017|2     |5.367142857142857 |5.198571428571428 |19.75             |5.6               |5.654285714285714 |
|Ajo Criollo O Napuri|2017|3     |4.9628571428571435|5.367142857142857 |19.571428571428573|0.7               |5.555714285714286 |
|Ajo Criollo O Napuri|2017|4     |5.654285714285714 |4.9628571428571435|20.571428571428573|13.299999999999999|5.7457142857142856|
|Ajo Criollo O Napuri|2017|5 

In [61]:
try:
    # Definimos la ruta para la matriz final
    final_parquet_path = os.path.join(PLATA_PATH_GCS, 'matriz_aprendizaje_forecasting')

    # Guardamos df_features (Spark DataFrame) en formato Parquet
    # Usamos overwrite para permitir actualizaciones si volvemos a correr el proceso
    df_features.write.mode('overwrite')\
        .parquet(final_parquet_path)

    print(f'ETL FINALIZADO CON ÉXITO')
    print(f'Matriz de aprendizaje guardada en: {final_parquet_path}')
    print(f'Registros totales exportados: {df_features.count():,}')
except Exception as e:
    print(f'Error al guardar la matriz final: {e}')

ETL FINALIZADO CON ÉXITO
Matriz de aprendizaje guardada en: gs://food-price-peru-2025-01/plata/precios_clima_integrado/matriz_aprendizaje_forecasting
Registros totales exportados: 4,697
